In [ ]:
from google.colab import files

# Ti verrà chiesto di scegliere un file dal tuo computer.
# Seleziona il file kaggle.json appena scaricato.
uploaded = files.upload()

In [ ]:
# Crea la cartella nascosta .kaggle
!mkdir -p ~/.kaggle

# Sposta il file json nella cartella appena creata
!cp kaggle.json ~/.kaggle/

# Cambia i permessi del file per motivi di sicurezza
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Scarica il dataset indicando l'utente/nome-dataset
!kaggle datasets download -d hrishitpatil/flight-data-2024

# Estrai il file zip scaricato
!unzip flight-data-2024.zip

In [ ]:
import pandas as pd
import numpy as np

# 1. Caricamento ottimizzato: leggiamo SOLO le colonne necessarie.
# Questo salva moltissima RAM, vitale per un dataset di 7 milioni di righe!
# Sostituisci il path con il percorso corretto del tuo file (es. '/content/drive/MyDrive/flight_dataset_2024.csv')
file_path = 'flight_data_2024.csv'
colonne_selezionate = ['op_unique_carrier', 'origin', 'arr_delay', 'cancelled', 'month']

print("Caricamento del dataset in corso...")
df = pd.read_csv(file_path, usecols=colonne_selezionate)
print("Caricamento completato!\n")

# 2. Esplorazione Base
print("--- Informazioni sul dataset ---")
df.info()

print("\n--- Prime 5 righe ---")
display(df.head())

# 3. Valutazione Valori Nulli e Unici
print("\n--- Valori Nulli per colonna ---")
print(df.isnull().sum())

print("\n--- Valori Unici per colonna ---")
print(df.nunique())

# 4. Riflessioni e Pulizia (Data Cleaning)
# Molto probabilmente troverai dei valori nulli in 'arr_delay'.
# Questo accade perché se un volo è stato cancellato (cancelled == 1),
# non ha mai generato un ritardo di arrivo.
print("\n--- Gestione Valori Nulli ---")

# Creiamo una copia per la pulizia
df_clean = df.copy()

# Per i calcoli futuri (ritardo min, max, medio), avere NaN in arr_delay può dare problemi.
# Possiamo forzare arr_delay a 0 per i voli cancellati, oppure lasciarli NaN e gestirli
# poi nel codice MapReduce/Spark ignorandoli nei calcoli delle medie dei ritardi.
# Per ora, rimuoviamo solo eventuali righe che non hanno info su compagnia o origine (record corrotti).
df_clean = df_clean.dropna(subset=['op_unique_carrier', 'origin', 'month'])

# 5. Salvataggio del dataset ridotto
output_path = 'dataset_job_3_1_pulito.csv'
df_clean.to_csv(output_path, index=False)
print(f"\nDataset ridotto e preparato salvato con successo in: {output_path}")

In [ ]:
# 1. Isiliamo i record che hanno 'arr_delay' nullo
voli_senza_ritardo = df[df['arr_delay'].isnull()]

print("--- Analisi dei record con 'arr_delay' nullo ---")
print(f"Totale record con arr_delay nullo: {len(voli_senza_ritardo)}")

# 2. Contiamo quanti di questi sono cancellati (1) e quanti no (0)
distribuzione_cancellati = voli_senza_ritardo['cancelled'].value_counts()
print("\nDistribuzione dello stato 'cancelled' per questi record:")
print(distribuzione_cancellati)

# 3. Analisi logica dei risultati
if 1 in distribuzione_cancellati:
    print(f"\n- {distribuzione_cancellati[1]} record nulli sono giustificati dalla cancellazione del volo.")

if 0 in distribuzione_cancellati:
    print(f"- ATTENZIONE: Ci sono {distribuzione_cancellati[0]} record con ritardo nullo MA NON cancellati.")
    print("  (Molto probabilmente si tratta di voli deviati o di errori di registrazione).")

# 4. Verifichiamo l'insieme totale dei voli cancellati per completezza
voli_cancellati = df[df['cancelled'] == 1]
print("\n--- Analisi inversa sui voli cancellati ---")
print(f"Totale voli cancellati nel dataset: {len(voli_cancellati)}")
print(f"Di cui con 'arr_delay' nullo: {voli_cancellati['arr_delay'].isnull().sum()}")

In [ ]:
import pandas as pd

# 1. Ricarichiamo dal file originale SOLO le colonne necessarie per la verifica
# (Sostituisci 'flight_data.csv' con il nome esatto del file estratto da Kaggle)
file_path_originale = 'flight_data_2024.csv'
colonne_verifica = ['arr_delay', 'cancelled', 'diverted']

print("Caricamento dati per la verifica in corso...")
df_verifica = pd.read_csv(file_path_originale, usecols=colonne_verifica)
print("Caricamento completato!\n")

# 2. Isoliamo il nostro gruppo "sospetto":
# Ritardo nullo MA NON cancellati
voli_sospetti = df_verifica[(df_verifica['arr_delay'].isnull()) & (df_verifica['cancelled'] == 0)]

print(f"Totale voli sospetti (nullo ma non cancellati): {len(voli_sospetti)}")

# 3. Verifichiamo quanti di questi sono effettivamente deviati (diverted == 1)
distribuzione_deviati = voli_sospetti['diverted'].value_counts()

print("\n--- Risultato della Verifica ---")
print("Stato 'diverted' per i voli sospetti:")
print(distribuzione_deviati)

# 4. Conclusione logica
if 1 in distribuzione_deviati and distribuzione_deviati[1] == len(voli_sospetti):
    print("\n✅ IPOTESI CONFERMATA: Tutti i 17.499 record sono voli deviati.")
    print("Possiamo procedere con tranquillità alla loro rimozione per il Job 3.1.")
elif 1 in distribuzione_deviati:
    print(f"\n⚠️ PARZIALMENTE CONFERMATA: {distribuzione_deviati[1]} sono voli deviati.")
    print(f"Tuttavia, rimangono {distribuzione_deviati.get(0, 0)} voli con ritardo nullo, non cancellati e non deviati (probabili errori sensore/dati mancanti).")
else:
    print("\n❌ IPOTESI SMENTITA: Nessuno di questi voli risulta deviato. Si tratta di record anomali o errori di registrazione.")

In [ ]:
# Manteniamo le colonne essenziali (puoi aggiungere 'diverted' se vuoi averlo esplicito,
# ma non è strettamente necessario se usi la logica dei null su arr_delay)
colonne_finali = ['op_unique_carrier', 'origin', 'arr_delay', 'cancelled', 'month']
df_final = df[colonne_finali].copy()

# Rimuoviamo solo eventuali righe gravemente incomplete (se esistono)
df_final = df_final.dropna(subset=['op_unique_carrier', 'origin', 'month', 'cancelled'])

# Salvataggio finale
output_path = 'dataset_job_3_1_definitivo.csv'
df_final.to_csv(output_path, index=False)
print(f"Dataset definitivo salvato! Totale righe: {len(df_final)}")

In [ ]:
import pandas as pd

print("Caricamento del dataset completo in corso...")
# Assicurati di inserire il nome corretto del file CSV originale da 7M di righe
df = pd.read_csv("flight_data_2024.csv")

# 1. Definizione delle colonne (Ho messo i nomi classici in minuscolo, modificali se nel tuo CSV sono in maiuscolo)
colonne_ideali = [
    "origin", "month", "dep_delay", "arr_delay", "cancelled",
    "cancellation_code", "carrier_delay", "weather_delay",
    "nas_delay", "security_delay", "late_aircraft_delay"
]

# Estraiamo solo le colonne che esistono effettivamente nel tuo dataset per evitare errori
colonne_esistenti = [col for col in colonne_ideali if col in df.columns]
df_job3_2 = df[colonne_esistenti].copy()
print(f"Colonne selezionate: {df_job3_2.columns.tolist()}")

# 2. Pulizia e Gestione dei Valori Nulli (NaN)

# Le cause di ritardo vuote significano che non c'è stato quel ritardo (-> 0)
cause_ritardo = ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]
for col in cause_ritardo:
    if col in df_job3_2.columns:
        df_job3_2[col] = df_job3_2[col].fillna(0.0)

# I codici di cancellazione vuoti significano che il volo non è cancellato (-> stringa vuota o 'N')
if "cancellation_code" in df_job3_2.columns:
    df_job3_2["cancellation_code"] = df_job3_2["cancellation_code"].fillna("N")

# Attenzione: Lasciamo i NaN su 'dep_delay' e 'arr_delay' volutamente per i voli cancellati!
# In questo modo, aggregatori come AVG() nei vari tool li ignoreranno in automatico senza falsare la media con degli zeri.

# 3. (Opzionale) Rimuoviamo righe corrotte che non hanno un aeroporto di origine o mese
df_job3_2 = df_job3_2.dropna(subset=["origin", "month"])

# 4. Salvataggio del nuovo dataset
nome_output = "dataset_job_3_2_definitivo.csv"
print(f"Salvataggio in {nome_output}...")
df_job3_2.to_csv(nome_output, index=False)
print("Fatto! Scarica il file e preparati per MapReduce/Hive/Spark.")

In [ ]:
import pandas as pd

print("Caricamento del dataset...")
# Sostituisci con il percorso in cui hai caricato il file su Colab
df = pd.read_csv("dataset_job_3_2_definitivo.csv")

print("\n" + "="*50)
print("ANALISI DEL DATASET JOB 3.2")
print("="*50)

# 1. Informazioni Generali
print(f"\nNumero totale di righe: {len(df):,}")
print(f"Numero totale di colonne: {len(df.columns)}")

# 2. Analisi dei Valori Nulli
print("\n--- VALORI NULLI ---")
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100

null_df = pd.DataFrame({
    'Valori Nulli': null_counts,
    'Percentuale (%)': null_percentages.round(2)
})
print(null_df)

# 3. Analisi dei Valori Unici (solo per colonne specifiche)
print("\n--- VALORI UNICI E STATISTICHE ---")

# Aeroporti di origine unici
if 'origin' in df.columns:
    print(f"Numero di aeroporti di origine unici: {df['origin'].nunique()}")

# Mesi unici
if 'month' in df.columns:
    mesi_unici = sorted(df['month'].dropna().unique().tolist())
    print(f"Mesi presenti nel dataset: {mesi_unici}")

# Codici di cancellazione unici
if 'cancellation_code' in df.columns:
    codici = df['cancellation_code'].dropna().unique().tolist()
    print(f"Codici di cancellazione presenti: {codici}")
    print("Distribuzione delle cancellazioni:")
    print(df['cancellation_code'].value_counts())

# Valori booleani per voli cancellati
if 'cancelled' in df.columns:
    print("\nDistribuzione colonna 'cancelled' (0=Regolare, 1=Cancellato):")
    print(df['cancelled'].value_counts())

print("\n" + "="*50)
print("Analisi completata.")

In [ ]:
import pandas as pd
import gc
import os

print("=== INIZIO ELABORAZIONE JOB 3.1 ===")
file_31 = 'dataset_job_3_1_definitivo.csv'

# 1. Lettura del dataset originale
print("Lettura dataset 3.1 in corso...")
df_31 = pd.read_csv(file_31)

# 2. Creazione Campione Ridotto (1%)
print("Creazione campione ridotto (1%)...")
df_31_ridotto = df_31.sample(frac=0.01, random_state=42) # random_state garantisce la riproducibilità
df_31_ridotto.to_csv('dataset_job_3_1_ridotto.csv', index=False)
del df_31_ridotto # Liberiamo subito la RAM
gc.collect()

# 3. Creazione Campione Aumentato (10x)
print("Creazione campione aumentato (10x)...")
file_out_31 = 'dataset_job_3_1_aumentato.csv'
# Scriviamo solo l'intestazione per inizializzare il file
df_31.head(0).to_csv(file_out_31, index=False)

# Appendiamo il dataframe 10 volte su disco
for i in range(10):
    df_31.to_csv(file_out_31, mode='a', header=False, index=False)
    print(f" - Blocco {i+1}/10 scritto.")

# Pulizia totale RAM prima di passare al prossimo Job
del df_31
gc.collect()
print("Job 3.1 completato!\n")


print("=== INIZIO ELABORAZIONE JOB 3.2 ===")
file_32 = 'dataset_job_3_2_definitivo.csv'

# 1. Lettura del dataset originale
print("Lettura dataset 3.2 in corso...")
df_32 = pd.read_csv(file_32)

# 2. Creazione Campione Ridotto (1%)
print("Creazione campione ridotto (1%)...")
df_32_ridotto = df_32.sample(frac=0.01, random_state=42)
df_32_ridotto.to_csv('dataset_job_3_2_ridotto.csv', index=False)
del df_32_ridotto
gc.collect()

# 3. Creazione Campione Aumentato (5x)
print("Creazione campione aumentato (5x)...")
file_out_32 = 'dataset_job_3_2_aumentato.csv'
# Scriviamo l'intestazione
df_32.head(0).to_csv(file_out_32, index=False)

# Appendiamo il dataframe 5 volte su disco
for i in range(5):
    df_32.to_csv(file_out_32, mode='a', header=False, index=False)
    print(f" - Blocco {i+1}/5 scritto.")

del df_32
gc.collect()
print("Job 3.2 completato!\n")

# Stampiamo le dimensioni finali dei file generati per controllo
print("=== DIMENSIONI FILE GENERATI ===")
def print_size(filename):
    size_mb = os.path.getsize(filename) / (1024 * 1024)
    print(f"{filename}: {size_mb:.2f} MB")

print_size('dataset_job_3_1_ridotto.csv')
print_size('dataset_job_3_1_aumentato.csv')
print_size('dataset_job_3_2_ridotto.csv')
print_size('dataset_job_3_2_aumentato.csv')